# Day 15 Assignment:Talk to Your Data: Building a Natural Language to SQL Generator

Name: Divyashree R

### Objective
To build a Natural Language to SQL system that allows users to query employee data using plain English.

The system converts natural language questions into SQL queries using Google's Gemini model and executes them on a dataset.

Dataset: Employee Information, created using Mockaroo

Model Used: Gemini API

In [ ]:
!pip install google-generativeai

In [ ]:
import pandas as pd
import sqlite3
import google.generativeai as genai

In [ ]:
df = pd.read_csv("employees.csv")

df.head()

,employee_id,first_name,last_name,gender,date_of_birth,job_title,department,salary,hire_date,email
0,1,Herrick,Mullarkey,Male,5/19/1986,Assistant Professor,Services,55993.34,12/1/2015,hmullarkey0@ezinearticles.com
1,2,Westley,Pietruschka,Male,3/12/1967,Civil Engineer,Legal,110296.78,3/31/2011,wpietruschka1@google.com.au
2,3,Boothe,Ickowics,Male,7/23/1998,Research Assistant III,Product Management,65972.98,12/13/2016,bickowics2@pcworld.com
3,4,Wandis,Tym,Female,6/29/1990,Help Desk Technician,Legal,57730.73,7/23/2010,wtym3@constantcontact.com
4,5,Berte,Ogglebie,Female,6/11/1985,Senior Financial Analyst,Business Development,111200.24,8/12/2016,bogglebie4@amazonaws.com


In [ ]:
connection = sqlite3.connect("employee.db")
cursor = connection.cursor()

In [ ]:
df.to_sql(
    name="employees",
    con=connection,
    if_exists="replace",
    index=False
)

print("Table created successfully")

Table created successfully


In [ ]:
query = "SELECT * FROM employees LIMIT 5"

pd.read_sql(query, connection)

,employee_id,first_name,last_name,gender,date_of_birth,job_title,department,salary,hire_date,email
0,1,Herrick,Mullarkey,Male,5/19/1986,Assistant Professor,Services,55993.34,12/1/2015,hmullarkey0@ezinearticles.com
1,2,Westley,Pietruschka,Male,3/12/1967,Civil Engineer,Legal,110296.78,3/31/2011,wpietruschka1@google.com.au
2,3,Boothe,Ickowics,Male,7/23/1998,Research Assistant III,Product Management,65972.98,12/13/2016,bickowics2@pcworld.com
3,4,Wandis,Tym,Female,6/29/1990,Help Desk Technician,Legal,57730.73,7/23/2010,wtym3@constantcontact.com
4,5,Berte,Ogglebie,Female,6/11/1985,Senior Financial Analyst,Business Development,111200.24,8/12/2016,bogglebie4@amazonaws.com


In [ ]:
import pandas as pd

departments = pd.DataFrame({
    "department_id": [1,2,3,4,5],
    "department_name": ["HR","Engineering","Sales","Marketing","Finance"]
})

departments.to_sql("departments", connection, if_exists="replace", index=False)

print("Departments table created successfully!")

Departments table created successfully!


In [ ]:
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('employees',), ('departments',)]


In [ ]:
from google import genai
from google.colab import userdata

In [ ]:
genai_client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

In [ ]:
prompt = """
### ROLE
You are a world-class Data Engineer and SQL expert.

Your job is to convert natural language questions into accurate SQL queries.

### CONTEXT
The user wants to query an employee database using natural language.

The database contains one table called **employees** with the following columns:

employee_id
first_name
last_name
email
department
salary
hire_date
city

The database is stored in SQLite.

### TASK
Your task is to:

1. Understand the user's natural language question.
2. Identify which columns are required.
3. Generate a correct SQL query to answer the question.
4. Ensure the SQL query is valid for SQLite.

### CONSTRAINTS
Follow these rules strictly:

- Only use the table **employees**
- Only use the columns listed above
- Do NOT invent columns
- Do NOT include explanations
- Do NOT include markdown
- Return ONLY the SQL query
- The SQL query must be executable in SQLite

### EXAMPLES

Example 1
Question: Show all employees working in the HR department.

SQL:
SELECT * FROM employees WHERE department = 'HR';


Example 2
Question: What is the average salary of employees in the IT department?

SQL:
SELECT AVG(salary) FROM employees WHERE department = 'IT';


Example 3
Question: List employees hired after 2021.

SQL:
SELECT * FROM employees WHERE hire_date > '2021-01-01';


Example 4
Question: Which city has the highest number of employees?

SQL:
SELECT city, COUNT(*) AS employee_count
FROM employees
GROUP BY city
ORDER BY employee_count DESC
LIMIT 1;

### OUTPUT FORMAT
Return ONLY the SQL query.

### USER QUESTION
{question}
"""

In [ ]:
def generate_sql(question):

    full_prompt = prompt.format(question=question)

    response = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=full_prompt
    )

    sql_query = response.text.strip()

    return sql_query

In [ ]:
def run_sql_query(query):

    try:
        result = pd.read_sql(query, connection)
        return result
    except Exception as e:
        return str(e)

In [ ]:
questions = [
    "List employees with salary greater than 60000",
    "What is the average salary?"
]

for q in questions:
    print("\nQuestion:", q)
    sql_query = generate_sql(q)
    print("SQL:", sql_query)
    display(run_sql_query(sql_query))


Question: ist employees with salary greater than 60000
SQL: SELECT * FROM employees WHERE salary > 60000;


,employee_id,first_name,last_name,gender,date_of_birth,job_title,department,salary,hire_date,email
0,2,Westley,Pietruschka,Male,3/12/1967,Civil Engineer,Legal,110296.78,3/31/2011,wpietruschka1@google.com.au
1,3,Boothe,Ickowics,Male,7/23/1998,Research Assistant III,Product Management,65972.98,12/13/2016,bickowics2@pcworld.com
2,5,Berte,Ogglebie,Female,6/11/1985,Senior Financial Analyst,Business Development,111200.24,8/12/2016,bogglebie4@amazonaws.com
3,6,Vanni,Romanetti,Female,11/24/1953,Help Desk Technician,Sales,147815.41,2/26/2018,vromanetti5@sphinn.com
4,8,Lillian,Kegley,Female,3/22/1959,Analyst Programmer,Product Management,98672.31,2/9/2015,lkegley7@disqus.com
...,...,...,...,...,...,...,...,...,...,...
148,193,Far,Strete,Male,10/27/1972,Research Nurse,Accounting,62421.08,11/24/2011,fstrete5c@prweb.com
149,194,Shalne,Burnsides,Female,7/19/1974,Electrical Engineer,Research and Development,149277.54,9/8/2015,sburnsides5d@altervista.org
150,195,Basia,Losano,Female,6/8/1979,Paralegal,Engineering,70254.77,1/15/2014,blosano5e@list-manage.com
151,198,William,Eckh,Male,11/27/1982,Analog Circuit Design manager,Support,121380.28,1/29/2018,weckh5h@nifty.com



Question: What is the average salary?
SQL: SELECT AVG(salary) FROM employees;


,AVG(salary)
0,89228.46045


In [ ]:
q = "Show all employees in the HR department"

print("\nQuestion:", q)

sql_query = generate_sql(q)

print("SQL:", sql_query)

display(run_sql_query(sql_query))


Question: Show all employees in the HR department
SQL: SELECT * FROM employees WHERE department = 'HR'


,employee_id,first_name,last_name,gender,date_of_birth,job_title,department,salary,hire_date,email


In [ ]:
q = "Show employees with their department names"

print("\nQuestion:", q)

sql_query = generate_sql(q)

print("SQL:", sql_query)

display(run_sql_query(sql_query))


Question: Show employees with their department names
SQL: SELECT first_name, last_name, department FROM employees


,first_name,last_name,department
0,Herrick,Mullarkey,Services
1,Westley,Pietruschka,Legal
2,Boothe,Ickowics,Product Management
3,Wandis,Tym,Legal
4,Berte,Ogglebie,Business Development
...,...,...,...
195,Carolus,Peck,Training
196,Ilsa,Scholes,Support
197,William,Eckh,Support
198,Sean,Laurentin,Legal


## Conclusion

In this assignment, we built a **Natural Language to SQL system** that allows users to query a database using simple English questions.

An employee dataset was first created and stored in a **SQLite database**. Then, we integrated the **Google Gemini (gemini-2.5-flash) model** to convert natural language questions into SQL queries. The model uses the provided database schema and prompt instructions to generate appropriate SQL statements.

The workflow of the system is:

**Natural Language Question → Gemini Model → Generated SQL Query → SQL Execution → Query Results**

We tested the system with multiple questions such as retrieving employees with salaries greater than a certain value and calculating aggregate statistics like the average salary. The generated SQL queries were executed on the SQLite database and the results were displayed using pandas.

